# Part 1: 양자화와 다양한 데이터 타입 - 실습

- 실습 1: 비트와 데이터 표현의 기초
- 실습 2: 부동소수점 정밀도 이해하기
- 실습 3: 모델 메모리 크기 계산하기

In [1]:
# ======================================================================================
# [실습 목표] Part 1: 양자화(Quantization)를 위한 기초 체력 다지기
# ======================================================================================
#
# 1. [Why] 왜 이 짓을 하는가? (필요성)
#    - "모델을 4bit로 줄인다"는 말을 이해하려면, 먼저 "원래 몇 bit인지", "bit가 뭔지" 알아야 합니다.
#    - FP32, FP16, BF16, INT8 등 딥러닝에서 쓰이는 외계어 같은 용어들의 실체를 해부합니다.
#    - 왜 숫자를 줄이면 계산이 틀리는지(오차), 왜 어느 순간 값이 안 변하는지 눈으로 직접 확인합니다.
#
# 2. [Goal] 무엇을 확인하는가? (목표)
#    - 데이터 타입별로 메모리를 얼마나 잡아먹는지 직접 잽니다. (FP32는 4바이트, FP16은 2바이트...)
#    - "컴퓨터는 0.1 + 0.2를 0.3이라고 계산하지 못한다"는 충격적인 사실을 확인합니다.
#    - 실제 모델(MLP, GPT)의 파라미터 개수를 가지고 "내 그래픽 카드에 들어갈까?"를 계산하는 법을 배웁니다.
#
# 3. [Key Tech] 핵심 기술 용어
#    - Bit(비트): 0 또는 1을 저장하는 정보의 최소 단위.
#    - Precision(정밀도): 소수점 아래 몇 번째 자리까지 정확하게 표현할 수 있는가?
#    - Overflow(오버플로우): 그릇(비트 수)이 너무 작아서 숫자가 넘쳐흐르는 현상.
#    - Machine Epsilon(머신 엡실론): 1.0과 그 다음으로 표현 가능한 숫자 사이의 간격. (이보다 작은 차이는 무시됨)
#
# ======================================================================================

import torch
import numpy as np

# [기계적 구동] 
# torch 라이브러리를 메모리에 로드하고 버전을 확인합니다. 
# 버전(예: 2.1.0+cu121)에 따라 지원하는 데이터 타입(FP8 등)이 다를 수 있어 확인이 필요합니다.
print(f"PyTorch 버전: {torch.__version__}")


PyTorch 버전: 2.10.0.dev20251029+cu130


---
## 실습 1: 비트와 데이터 표현의 기초

### 학습 목표
- 다양한 데이터 타입의 비트 수와 표현 범위 이해
- Signed vs Unsigned 정수의 차이 이해
- 오버플로우 현상 직접 확인

### 1-1. 데이터 타입별 메모리 크기 확인

In [ ]:
# [설명] 실험할 부동소수점(실수) 타입들의 리스트를 만듭니다.
# 튜플 형태: ("이름", 파이토치_타입_상수)
float_types = [
    ("FP64 (double)", torch.float64), # 64비트: 엄청 정밀하지만 용량 2배 (과학 계산용)
    ("FP32 (float)", torch.float32),  # 32비트: 딥러닝의 표준 (Standard)
    ("FP16 (half)", torch.float16),   # 16비트: 용량 절반, 속도 빠름 (단, 오버플로우 위험)
    ("BF16 (bfloat16)", torch.bfloat16), # 16비트: 구글이 만든 AI 전용 포맷 (숫자 범위가 넓음)
]

print("=" * 60)
print("부동소수점 타입 비교")
print("=" * 60)

# [실험] 파이(π)라는 무한소수를 각기 다른 크기의 그릇(데이터 타입)에 담아봅니다.
for name, dtype in float_types:
    # [기계적 구동]
    # 1. 3.1415... 라는 파이썬 float 객체(기본 64bit)를 가져옵니다.
    # 2. 지정된 dtype(예: FP16) 형식으로 강제 변환(Casting)하여 텐서를 생성합니다.
    #    -> 이 과정에서 그릇 크기에 맞게 소수점 아래 숫자들이 잘려나갑니다(Truncation).
    tensor = torch.tensor([3.14159265358979], dtype=dtype)
    
    # element_size(): 텐서 하나가 몇 바이트를 차지하는지 운영체제에게 물어봅니다.
    print(f"{name:20} | 크기: {tensor.element_size()}바이트 | 비트: {tensor.element_size() * 8}bit | 값: {tensor.item():.10f}")



부동소수점 타입 비교
FP64 (double)        | 크기: 8바이트 | 비트: 64bit | 값: 3.1415926536
FP32 (float)         | 크기: 4바이트 | 비트: 32bit | 값: 3.1415927410
FP16 (half)          | 크기: 2바이트 | 비트: 16bit | 값: 3.1406250000
BF16 (bfloat16)      | 크기: 2바이트 | 비트: 16bit | 값: 3.1406250000


# 🧪 [실험 결과] 데이터 타입별 정밀도와 메모리 반비례 관계

이번 실험은 **"그릇(Bit)의 크기가 작아질 때 데이터의 디테일(정밀도)이 어떻게 파괴되는가"**를 보여줍니다.

---

## 1. 타입별 정밀도 분석 표

| 타입  | Bit | 크기(Byte) | π(파이) 값 결과 | 정밀도 수준 |
|-------|-----|------------|-----------------|-------------|
| FP64  | 64  | 8 Byte     | 3.1415926536    | [완벽] 원본 손실 거의 없음 |
| FP32  | 32  | 4 Byte     | 3.1415927410    | [우수] 끝자리만 살짝 변함 (표준) |
| FP16  | 16  | 2 Byte     | 3.1406250000    | [위험] 소수점 셋째 자리부터 뭉개짐 |
| BF16  | 16  | 2 Byte     | 3.1406250000    | [특수] FP16과 같으나 학습 안정성 강화 |

---

## 2. 🍱 핵심 해석: '반찬통' 비유로 본 정밀도 손실

- **왕대자 반찬통 (FP64)**  
  → 64cm짜리 긴 국수를 다 담을 수 있어서 원 모양 그대로 보존됩니다. 하지만 냉장고 공간을 너무 많이 차지합니다.

- **표준 반찬통 (FP32)**  
  → 가장 흔한 32cm 통입니다. 국수 끝이 아주 조금 잘리지만 먹는 데(연산) 지장이 없습니다.

- **다이어트 도시락 (FP16/BF16)**  
  → 공간을 아끼려고 16cm 통에 억지로 담았습니다. 국수의 절반 이상이 잘려나갔습니다. (값의 오차 발생)

---

## 3. 💡 AI 모델은 왜 "망가진 데이터"를 쓸까요?

- **정밀도 손실 (Trade-off)**  
  데이터를 16bit나 4bit로 줄이면 숫자들은 뭉개지지만, 그 대가로 메모리(VRAM) 공간을 4배~16배 확보할 수 있습니다.

- **결론**  
  소수점 아래 열 번째 자리까지 정확한 모델 1개를 돌리는 것보다, 조금 덜 정확하더라도 **4배 더 많은 지식(매개변수)**을 가진 모델을 돌리는 것이 AI 지능 면에서는 훨씬 유리합니다.

---

## 📌 핵심 요약

LLM 양자화를 한다는 것은  
**"소수점 아래의 정밀한 디테일을 버리고, 메모리 효율성과 실행 속도를 취하는 경제적인 선택"**을 하는 것입니다.

In [3]:
# [설명] 이번엔 정수(Integer) 타입들입니다. 소수점이 없는 숫자들입니다.
int_types = [
    ("INT64 (long)", torch.int64),    # 가장 큰 정수 (인덱싱용)
    ("INT32 (int)", torch.int32),     # 일반적인 정수
    ("INT16 (short)", torch.int16),   # 작은 정수
    ("INT8 (char)", torch.int8),      # 아주 작은 정수 (-128 ~ 127) -> 양자화의 핵심!
    ("UINT8 (unsigned)", torch.uint8),# 부호 없는 8비트 정수 (0 ~ 255) -> 이미지 픽셀용
]

print("=" * 60)
print("정수 타입 비교")
print("=" * 60)

# [실험] 정수 42를 각 타입에 담아 메모리 사용량을 확인합니다.
for name, dtype in int_types:
    tensor = torch.tensor([42], dtype=dtype)
    print(f"{name:20} | 메모리 크기: {tensor.element_size()}바이트 | 비트: {tensor.element_size() * 8}bit")

정수 타입 비교
INT64 (long)         | 메모리 크기: 8바이트 | 비트: 64bit
INT32 (int)          | 메모리 크기: 4바이트 | 비트: 32bit
INT16 (short)        | 메모리 크기: 2바이트 | 비트: 16bit
INT8 (char)          | 메모리 크기: 1바이트 | 비트: 8bit
UINT8 (unsigned)     | 메모리 크기: 1바이트 | 비트: 8bit


# 🔢 정수(Integer) 타입 해석: "용량 다이어트의 끝판왕"

---

## 1. INT64 ~ INT16 (여유로운 정수들)

- **INT64 (8바이트)**  
  → 숫자를 담는 그릇이 엄청나게 큽니다. 우주 전체의 별 개수만큼 큰 숫자도 담을 수 있지만, 그만큼 메모리를 많이 잡아먹습니다.

- **INT32 (4바이트)**  
  → 우리가 프로그래밍할 때 가장 흔히 쓰는 평범한 정수 그릇입니다.

---

## 2. INT8 (양자화의 핵심 - 1바이트)

- **메모리 크기**: 1바이트 (8bit)  
- **해석**: 불필요한 장식을 다 떼어내고 딱 -128부터 127까지만 담는 아주 작은 그릇입니다.  
- **핵심 포인트**: AI 모델의 복잡한 실수(예: 3.1415...)를 이 작은 INT8 정수로 변환하는 것이 바로 **양자화(Quantization)**입니다.  
- **효과**: 16비트 실수(2바이트)를 8비트 정수(1바이트)로 바꾸면, 메모리 사용량이 즉시 절반으로 줄어듭니다!

---

## 3. UINT8 (이미지/픽셀용 - 1바이트)

- **해석**: U는 **Unsigned(부호 없음)**을 뜻합니다. 마이너스를 포기하는 대신 0부터 255까지 담을 수 있습니다.  
- **실제 사례**: 디지털 사진의 RGB 색상이 0~255 단계로 되어 있는 이유가 바로 이 **UINT8**을 쓰기 때문입니다.

---

## 💡 왜 양자화에서 '정수(Int)'를 쓸까요?

- **계산 단순화**  
  → 컴퓨터에게 "실수(Float) 더하기"는 머리 아픈 일이지만, **"정수(Int) 더하기"**는 눈 감고도 할 수 있는 아주 쉬운 일입니다. (연산 속도 상승!)

- **하드웨어 최적화**  
  → 최신 그래픽카드(NVIDIA Tensor Core 등)는 이 **INT8 연산**을 광속으로 처리하도록 설계되어 있습니다.

---

## 🏁 최종 결론

정수 타입으로 내려올수록 데이터는 가벼워지고 연산은 단순해집니다.  
특히 **8비트(INT8)**는 똑똑함을 어느 정도 유지하면서 **용량과 속도를 모두 잡을 수 있는 '양자화의 달콤한 스팟(Sweet Spot)'**입니다!

### 1-2. Signed vs Unsigned 정수 표현 범위

In [4]:
# [설명] Signed INT8은 맨 앞 1비트를 부호(+, -)를 표시하는 데 씁니다. (MSB: Most Significant Bit)
# 그래서 숫자를 표현하는 공간이 7비트밖에 안 남아 범위가 반토막 납니다. (-128 ~ 127)
print("INT8 (Signed 8-bit):")
# torch.iinfo(): 정수 타입의 정보(Information)를 조회하는 함수
print(f"  최소값: {torch.iinfo(torch.int8).min}")
print(f"  최대값: {torch.iinfo(torch.int8).max}")
print(f"  표현 가능한 값의 개수: {2**8} (= 256개)")

print()

# [설명] Unsigned INT8은 부호 비트가 필요 없어서 8비트를 모두 숫자 표현에 씁니다.
# 그래서 0부터 255까지 표현 가능합니다. (음수 표현 불가)
print("UINT8 (Unsigned 8-bit):")
print(f"  최소값: {torch.iinfo(torch.uint8).min}")
print(f"  최대값: {torch.iinfo(torch.uint8).max}")
print(f"  표현 가능한 값의 개수: {2**8} (= 256개)")

INT8 (Signed 8-bit):
  최소값: -128
  최대값: 127
  표현 가능한 값의 개수: 256 (= 256개)

UINT8 (Unsigned 8-bit):
  최소값: 0
  최대값: 255
  표현 가능한 값의 개수: 256 (= 256개)


# 📦 '256개의 칸이 있는 상자' 비유

우리에겐 딱 **256칸이 있는 마법의 상자**가 있습니다.  
(8비트 = \(2^8 = 256\))

---

## 1. INT8 (Signed - "부호가 있는 정수")

- **전략**: "세상엔 마이너스(-)도 필요하잖아?"  
- **방법**: 맨 첫 번째 칸을 숫자가 아닌 **「+, - 표시판」**으로 사용합니다. (MSB)  
- **결과**: 숫자를 담을 칸이 줄어들어 **-128 ~ +127**까지만 표현 가능합니다.  
- **특징**: 0을 기준으로 양쪽이 균형 잡혀 있습니다.  
  → **AI의 가중치(Weight)**는 보통 0을 중심으로 플러스/마이너스가 골고루 나오기 때문에 양자화에서 매우 중요합니다.

---

## 2. UINT8 (Unsigned - "부호가 없는 정수")

- **전략**: "난 마이너스 안 쓸 건데? 그냥 큰 숫자 담을래!"  
- **방법**: 표시판을 치워버리고 256칸 전체를 숫자로 채웁니다.  
- **결과**: **0 ~ 255**까지 표현 가능합니다.  
- **특징**: 마이너스가 필요 없는 **이미지 데이터(RGB 0~255)** 등을 다룰 때 가장 효율적입니다.

---

## 🔍 왜 이게 양자화에서 중요한가요?

# 📏 '256칸짜리 자' 비유

우리가 가진 자는 눈금이 딱 **256개뿐**입니다.  
(8비트 = \(2^8 = 256\))  
이 자로 **0cm부터 10cm까지**를 잰다고 상상해보세요.

---

## 상황 B: 데이터가 전부 양수(0~10)뿐이라면? → **UINT8 유리**

- **UINT8 선택**: 자의 256개 눈금을 모두 0~10cm를 나누는 데 사용  
- **눈금 한 칸 간격**:  
  \[
  10cm \div 256 \approx \mathbf{0.039cm}
  \]

- **INT8 선택**: 자의 절반(128칸)은 마이너스를 위해 비워둠 → 남은 128칸으로 0~10cm를 나눔  
- **눈금 한 칸 간격**:  
  \[
  10cm \div 128 \approx \mathbf{0.078cm}
  \]

- **해석**: UINT8을 쓰면 눈금이 훨씬 촘촘합니다.  
  → **"더 세밀하게(정밀하게) 표현할 수 있다"**는 뜻입니다.  
  (마이너스 칸을 낭비하지 않았으니까요!)

---

## 상황 A: 데이터가 음수와 양수(-5 ~ +5) 섞여 있다면? → **INT8 필수**

- **문제**: UINT8(0~255)로는 음수를 표현할 칸 자체가 없음  
- **해결**: 정밀도를 조금 포기하더라도, INT8을 써서  
  - 자의 왼쪽 절반 → 마이너스  
  - 자의 오른쪽 절반 → 플러스  
  로 범위를 균형 있게 나눠야 합니다.

---

## 💡 AI 모델에서 왜 이게 고민일까요?

- **가중치(Weights)의 특성)**  
  → 대부분 0을 기준으로 양수와 음수가 골고루 퍼져 있음 → **INT8 사용**

- **활성화 함수(Activation)의 특성)**  
  → ReLU 같은 함수는 음수를 다 지워버리고 0 이상의 값만 출력 → **UINT8 사용 시 더 촘촘하고 정확**

---

## 🏁 최종 요약

- **정밀도의 낭비 방지**: 데이터에 없는 범위(음수)를 위해 자의 절반을 비워두는 건 아까움  
- **촘촘함의 승리**: 데이터 성격에 맞춰 타입을 고르면, 똑같은 8비트라도  
  → 더 잘게 쪼개서 원본에 가깝게(정밀하게) 저장 가능

## 💡 한 줄 요약

**"똑같은 256칸이지만, '표시판(부호)'을 달아서 음수까지 챙길 것인가(INT8),  
아니면 표시판 떼고 양수만 더 멀리 갈 것인가(UINT8)의 차이일 뿐, 전체 용량은 8비트로 동일하다!"**

### 🎯 실습 1-1: FP16 정밀도 부족을 직접 확인하기 (1씩 증가가 안 되는 순간)
- 어느 순간부터 +1 해도 값이 안 바뀌는 현상이 발생함
- FP16은 비트 수가 작아서(16bit) 표현할 수 있는 실수 값이 제한적이다. 표현 가능한 숫자가 드문드문 존재함. 그래서 숫자가 연속적으로 저장되지 않고, 띄엄띄엄(계단처럼) 저장된다.
- 값이 커질수록 표현 가능한 숫자 사이 간격(step)이 커진다. 그 결과 어떤 구간에서는 +1 정도 변화가 너무 작아 반올림되어 값이 안 바뀐다.

In [5]:
import torch

# [실험 설정] FP16으로 2048.0을 만듭니다.
# 왜 2048인가? FP16에서 2048을 넘어가면 정밀도가 급격히 떨어지는 구간(지수부가 바뀌는 지점)이기 때문입니다.
x = torch.tensor([2048.0], dtype=torch.float16)
print("초기값:", x.item())

# 2048에서 1씩 늘려 보면서 실제 값이 바뀌는지 확인
# [기계적 구동] 
# CPU는 계산을 FP32 이상으로 하더라도, 결과를 다시 x(FP16)에 담을 때 '강제 반올림'이 일어납니다.
# 2048 다음으로 표현 가능한 FP16 숫자는 2050일 수도 있습니다. (간격이 2라면)
# 그러면 2049를 넣으려고 해도 가장 가까운 2048이나 2050으로 튕겨져 나갑니다.
for i in range(1, 20):
    y = x + i
    print(f"2048 + {i:2} = {y.item()}")

초기값: 2048.0
2048 +  1 = 2048.0
2048 +  2 = 2050.0
2048 +  3 = 2052.0
2048 +  4 = 2052.0
2048 +  5 = 2052.0
2048 +  6 = 2054.0
2048 +  7 = 2056.0
2048 +  8 = 2056.0
2048 +  9 = 2056.0
2048 + 10 = 2058.0
2048 + 11 = 2060.0
2048 + 12 = 2060.0
2048 + 13 = 2060.0
2048 + 14 = 2062.0
2048 + 15 = 2064.0
2048 + 16 = 2064.0
2048 + 17 = 2064.0
2048 + 18 = 2066.0
2048 + 19 = 2068.0


# 📏 FP16의 한계: '눈금이 망가진 자' 비유 + 수학적 원인

---

## 1. 근본 원인: 10비트의 가수부(Fraction)

- 컴퓨터는 실수를 **IEEE 754 과학적 표기법**  
  \[
  1.xxxx \times 2^e
  \]  
  으로 저장합니다.
- FP16의 구조: **[부호 1비트] + [지수 5비트] + [가수 10비트]**
- **가수부 10비트 한계**:  
  → 숫자를 최대 **1,024단계(\(2^{10}\))**로만 쪼갤 수 있음

---

## 2. 왜 2048에서 '1'이 무시되는가?

- **1024 ~ 2048 구간**  
  - 간격: 1024  
  - 1,024단계로 나누면 → 최소 단위 = **1.0**  
  - → 1씩 더하는 것이 가능

- **2048 ~ 4096 구간 (문제 구간)**  
  - 간격: 2,048  
  - 여전히 1,024단계뿐  
  - 계산:  
    \[
    2048 \div 1024 = \mathbf{2.0}
    \]  
  - → 최소 단위가 **2.0**으로 커짐

---

## 3. 결과 해석 (계단 현상)

- **2048 + 1 → 2048**  
  → 2049 눈금이 없음 → 가장 가까운 2048로 반올림  
- **2048 + 2 → 2050**  
  → 정확히 존재하는 눈금  
- **2048 + 3 → 2052**  
  → 2051 눈금 없음 → 2052로 점프  

👉 값이 **2048 → 2050 → 2052** 식으로 뚝뚝 끊겨 나오는 **계단 현상(Staircase Effect)** 발생

---

## 🔍 AI 학습에서의 문제 (FP16의 실제 영향)

### 1. 수치적 불안정성 (Numerical Instability)
- **학습의 본질**: 딥러닝은 가중치를 아주 미세하게 조정하면서 최적화합니다.  
  예: 0.0001 같은 작은 변화가 누적되어 모델이 점점 더 정밀해집니다.
- **FP16의 문제**: 큰 숫자 영역(예: 2048 이상)에서는 최소 단위가 2.0으로 커지므로,  
  1 이하의 작은 변화는 **표현할 수 없어 0으로 취급**됩니다.
- **결과**: 작은 오차가 반영되지 않아 학습이 멈추거나, 모델이 더 이상 개선되지 않는 현상이 발생합니다.

---

### 2. 계단 현상 (Staircase Effect)
- **연속성이 깨짐**: FP16은 특정 구간에서 숫자를 2048 → 2050 → 2052 식으로만 표현합니다.  
  → 중간 값(2049, 2051)은 아예 존재하지 않습니다.
- **학습 곡선 왜곡**: 모델은 원래 연속적인 패턴을 학습해야 하지만, 값이 계단처럼 뚝뚝 끊기면  
  → **부드러운 패턴 대신 거친 근사치**만 학습하게 됩니다.
- **비유**: 매끄러운 곡선을 따라가야 하는데, FP16에서는 계단식 그래프만 보고 배우는 셈입니다.

---

### 3. 그래디언트 문제와 연결
- **그래디언트 소실(Vanishing Gradient)**  
  작은 변화가 0으로 무시되면 역전파 과정에서 그래디언트가 점점 줄어들어 결국 **학습이 정지**합니다.
- **그래디언트 폭발(Exploding Gradient)**  
  반대로 값이 계단식으로 점프하면 그래디언트가 갑자기 커져서 **불안정한 학습**을 유발합니다.
- **결론**: FP16은 이 두 문제를 동시에 악화시킬 수 있어, 학습 안정성에 치명적입니다.

---

### 4. 학습 vs 추론의 차이
- **학습(Training)**: 미세한 변화까지 반영해야 하므로 FP32나 BF16 같은 정밀도가 높은 타입을 사용합니다.  
- **추론(Inference)**: 이미 학습된 모델을 실행만 할 때는 FP16을 써서 메모리와 속도를 절약합니다.

---

## 🏁 최종 요약

**"FP16은 가볍지만, 가수부가 10비트뿐이라  
2048을 넘는 순간 최소 단위가 2.0으로 커져버린다.  
그 결과 작은 숫자(1.0)는 기록할 공간이 없어 무시되거나 반올림되며,  
데이터가 계단처럼 끊겨버리는 '눈금이 망가진 자'가 된다."**

---
## 실습 2: 부동소수점 정밀도 이해하기

### 학습 목표
- FP32, FP16, BF16의 정밀도 차이 이해
- 부동소수점의 정밀도 한계 체험
- 양자화 전후 정밀도 손실 계산

### 2-1. 부동소수점 타입별 상세 정보

In [6]:
print("=" * 70)
print("FP32 (Single Precision) - 32비트")
print("=" * 70)
# torch.finfo(): 실수(float) 타입의 기계적 한계 정보를 조회합니다.
fp32_info = torch.finfo(torch.float32)
# Mantissa(가수) 비트가 많을수록 소수점 아래를 정밀하게 표현합니다.
print(f"  구조: 1 sign + 8 exponent + 23 mantissa bits")
print(f"  표현 범위: [{fp32_info.min:.2e}, {fp32_info.max:.2e}]")
print(f"  정밀도 (eps): {fp32_info.eps}")
print(f"  유효 자릿수: 약 7자리")

print()
print("=" * 70)
print("FP16 (Half Precision) - 16비트")
print("=" * 70)
fp16_info = torch.finfo(torch.float16)
# Exponent(지수) 비트가 5개밖에 없어서 표현할 수 있는 숫자의 크기 범위가 작습니다. (65504 넘으면 Infinity 됨)
print(f"  구조: 1 sign + 5 exponent + 10 mantissa bits")
print(f"  표현 범위: [{fp16_info.min:.2e}, {fp16_info.max:.2e}]")
print(f"  정밀도 (eps): {fp16_info.eps}")
print(f"  유효 자릿수: 약 3자리")

print()
print("=" * 70)
print("BF16 (Brain Float 16) - 16비트")
print("=" * 70)
bf16_info = torch.finfo(torch.bfloat16)
# [중요] BF16은 FP32에서 Mantissa만 잘라낸 형태입니다.
# 즉, 정밀도는 낮지만(소수점 아래는 대충), 표현 가능한 숫자의 크기 범위는 FP32와 같습니다.
# 그래서 학습 도중 숫자가 너무 커져서 터지는(Overflow) 일이 FP16보다 훨씬 적습니다.
print(f"  구조: 1 sign + 8 exponent + 7 mantissa bits")
print(f"  표현 범위: [{bf16_info.min:.2e}, {bf16_info.max:.2e}]")
print(f"  정밀도 (eps): {bf16_info.eps}")
print(f"  유효 자릿수: 약 2자리")
print(f"  특징: FP32와 같은 지수 범위, 딥러닝 학습에 최적화!")

FP32 (Single Precision) - 32비트
  구조: 1 sign + 8 exponent + 23 mantissa bits
  표현 범위: [-3.40e+38, 3.40e+38]
  정밀도 (eps): 1.1920928955078125e-07
  유효 자릿수: 약 7자리

FP16 (Half Precision) - 16비트
  구조: 1 sign + 5 exponent + 10 mantissa bits
  표현 범위: [-6.55e+04, 6.55e+04]
  정밀도 (eps): 0.0009765625
  유효 자릿수: 약 3자리

BF16 (Brain Float 16) - 16비트
  구조: 1 sign + 8 exponent + 7 mantissa bits
  표현 범위: [-3.39e+38, 3.39e+38]
  정밀도 (eps): 0.0078125
  유효 자릿수: 약 2자리
  특징: FP32와 같은 지수 범위, 딥러닝 학습에 최적화!


# 📏 FP32 vs FP16 vs BF16: "다이어트의 한계와 신개념 16비트"

---

## 1. FP32 vs FP16: "다이어트의 한계"

- **FP32 (32비트 부동소수점)**  
  - 지수부: 8비트  
  - 가수부: 23비트  
  - 특징: 숫자가 아무리 커져도 (약 \(10^{38}\)) 버티며, 소수점 아주 깊은 곳까지 정확하게 표현 가능  
  - 단점: 메모리 사용량이 크고 연산 속도가 느림  

- **FP16 (16비트 부동소수점)**  
  - 지수부: 5비트  
  - 가수부: 10비트  
  - 특징: 소수점 아래는 FP32보다 덜 정확하지만, 메모리 절약 효과 큼  
  - 치명적 단점: 숫자가 **65,504**를 넘어가면 **무한대(Inf)**로 처리 → 학습 중 Overflow 발생 시 학습 자체가 망가짐  

---

## 2. 🧠 BF16 (Brain Float 16): "신개념 16비트"

- **구글의 발명품**: 딥러닝에 특화된 포맷  
- **전략**:  
  - 지수부: FP32와 동일하게 8비트 → 큰 숫자도 안전하게 표현 가능  
  - 가수부: 7비트로 줄여서 정밀도는 희생  
- **결과**:  
  - 표현 범위: FP32와 동일 (\(10^{38}\)) → 학습 중 절대 Overflow 없음  
  - 정밀도: 유효 자릿수는 약 2자리 (FP16은 3자리) → 소수점 아래는 매우 거칠음  

---

## 🍱 핵심 해석: '망원경 vs 현미경' 비유

| 타입  | 구조 비유              | 한 줄 평가 |
|-------|------------------------|------------|
| FP32  | 성능 좋은 고화질 카메라 | 완벽함, 하지만 너무 무겁고 메모리 킬러 |
| FP16  | 저해상도 현미경        | 소수점 아래(가수부 10)는 BF16보다 상세하나, 큰 숫자엔 취약 (65,504 한계) |
| BF16  | 저해상도 망원경        | 소수점 아래(가수부 7)는 엉망이나, 먼 우주(\(10^{38}\))까지 안전하게 표현 가능 |

---

## 💡 왜 딥러닝에서 BF16이 대세가 되었나?

- **학습 안정성**: 작은 소수점 정밀도보다, 큰 숫자가 안전하게 표현되는 것이 훨씬 중요  
- **번거로움 제거**: FP16은 Overflow 방지를 위해 'Loss Scaling' 같은 복잡한 기법이 필요했지만, BF16은 그냥 써도 안전  
- **효율성**: FP32와 동일한 범위를 가지면서도 메모리 사용량은 절반(2바이트) → 학습 속도와 메모리 효율 극대화  

---

## 📌 추가 팁

- **공부할 때 (Training)**: **BF16 선호** → 안정적이고 Overflow 걱정 없음  
- **다 공부하고 실행할 때 (Inference)**: **FP16 선호** → 소수점 아래가 조금 더 정확해 결과가 매끄럽게 표현됨

### 2-2. 정밀도 손실 실험

In [7]:
# 파이 값을 다양한 정밀도로 표현
# 파이썬의 기본 float는 64비트(double)라 아주 정확합니다.
pi = 3.14159265358979323846

print("π (파이) 값의 정밀도 비교")
print("=" * 50)
print(f"실제 값:  {pi:.20f}")
print()

# 각기 다른 그릇에 담습니다.
pi_fp32 = torch.tensor([pi], dtype=torch.float32)
pi_fp16 = torch.tensor([pi], dtype=torch.float16)
pi_bf16 = torch.tensor([pi], dtype=torch.bfloat16)

# .item()으로 값을 꺼내서 비교해봅니다.
# BF16은 3.14까지만 맞고 뒤는 틀리는 것을 볼 수 있습니다.
print(f"FP32:     {pi_fp32.item():.20f}")
print(f"FP16:     {pi_fp16.item():.20f}")
print(f"BF16:     {pi_bf16.item():.20f}")

# 오차 계산 (절대값 차이)
print("\n오차 (|실제값 - 저장값|):")
print(f"FP32 오차: {abs(pi - pi_fp32.item()):.2e}")
print(f"FP16 오차: {abs(pi - pi_fp16.item()):.2e}")
print(f"BF16 오차: {abs(pi - pi_bf16.item()):.2e}")

π (파이) 값의 정밀도 비교
실제 값:  3.14159265358979311600

FP32:     3.14159274101257324219
FP16:     3.14062500000000000000
BF16:     3.14062500000000000000

오차 (|실제값 - 저장값|):
FP32 오차: 8.74e-08
FP16 오차: 9.68e-04
BF16 오차: 9.68e-04


# 🥧 π(파이) 값 정밀도 성적표 해석

---

## 1. FP32 (가장 모범적인 학생)

- **결과**: 3.1415927...  
- **오차**: \(8.74 \times 10^{-8}\) (0.0000000874)  
- **해석**: 실제 값(…6535)과 비교했을 때 **소수점 여섯째 자리까지 완벽하게 일치**합니다.  
- **평가**: 딥러닝에서 "이 정도면 충분하다"라고 보는 **표준적인 정밀도**입니다.

---

## 2. FP16 & BF16 (다이어트 중인 학생들)

- **결과**: 3.1406250... (둘 다 동일)  
- **오차**: \(9.68 \times 10^{-4}\) (0.000968)  
- **해석**: 소수점 둘째 자리(3.14)까지만 겨우 맞췄습니다.  
  셋째 자리인 1을 담으려 했지만, 그릇(가수부)이 부족해 가장 가까운 숫자인 3.1406...으로 반올림된 것입니다.  
- **오차 체감**: FP32보다 오차가 **약 10,000배나 큼!**

---

## 🔍 왜 FP16과 BF16의 결과가 똑같나요?

- **한계점 도달**: 3.1415...를 표현하기에는 FP16(10비트)이나 BF16(7비트) 모두 턱없이 부족했습니다.  
- **비유**: 100ml 컵(FP16)과 70ml 컵(BF16)에 10L의 물을 담으려는 상황.  
  → 둘 다 넘쳐버리고 겨우 한 모금만 남는데, 이번 실험에서는 그 '남은 한 모금'이 **3.140625**로 우연히 일치한 것.  
- **잠재적 위험**: 지금은 같아 보이지만, 긴 계산을 반복하면 FP16(10비트)이 BF16(7비트)보다 소수점 정밀도를 조금 더 잘 버텨줍니다.

---

## 💡 AI 학습에 미치는 영향

- **개별 오차는 크지만**: 숫자 하나가 틀리는 건 괜찮습니다.  
  AI는 수십억 개의 숫자가 서로 **"조율"**하며 결과를 내기 때문입니다.  
- **누적의 공포**: 하지만 학습을 수만 번 반복하면 작은 오차들이 누적되어  
  → 결국 모델이 **엉뚱한 대답**을 할 수 있습니다.  
- **실무 선택**: 그래서 학습(Training) 단계에서는 정밀도가 더 좋은 **BF16(범위가 넓어서 안전)**이나 FP32를 선호합니다.

---

## 🏁 최종 요약

**"16비트 타입(FP16/BF16)은 소수점 셋째 자리부터는 '내 알 바 아니다'라며 손을 놔버린다.  
하지만 그 대가로 우리는 메모리를 절반이나 아꼈으므로, 현대 AI는 이 억울한 오차를 기꺼이 받아들인다!"**

### 2-3. 부동소수점의 함정 - 0.1 + 0.2 != 0.3

In [8]:
# [설명] 컴퓨터 공학의 고전적인 문제입니다.
# 컴퓨터는 2진수(0, 1)만 쓰는데, 0.1(1/10)은 2진수로 표현하면 무한소수(0.00011001100...)가 됩니다.
# 그래서 정확히 0.1을 저장할 수 없고 '0.1에 아주 가까운 근사값'을 저장합니다.
a = torch.tensor([0.1], dtype=torch.float32)
b = torch.tensor([0.2], dtype=torch.float32)
c = torch.tensor([0.3], dtype=torch.float32)

result = a + b

print("0.1 + 0.2 == 0.3 ?")
print(f"0.1 = {a.item():.20f}")
print(f"0.2 = {b.item():.20f}")
# 두 근사값을 더했더니 오차가 누적되어 0.3과 미세하게 달라집니다.
print(f"0.1 + 0.2 = {result.item():.20f}")
print(f"0.3 = {c.item():.20f}")

# == 연산자로 비교하면 False가 나옵니다.
print(f"\n0.1 + 0.2 == 0.3: {(result == c).item()}")
# 아주 미세한 차이(엡실론)가 존재함을 확인합니다.
print(f"차이: {abs(result.item() - c.item()):.2e}")

0.1 + 0.2 == 0.3 ?
0.1 = 0.10000000149011611938
0.2 = 0.20000000298023223877
0.1 + 0.2 = 0.30000001192092895508
0.3 = 0.30000001192092895508

0.1 + 0.2 == 0.3: True
차이: 0.00e+00


물론이죠! 요청하신 내용을 깔끔하게 **Markdown 형식**으로 정리해드릴게요:

---

# 🔢 왜 0.1은 우리가 아는 0.1이 아닐까?

## 1. 근본 원인
- 컴퓨터는 모든 것을 **2진수(0, 1)**로 처리합니다.  
- 하지만 10진수 `0.1`을 2진수로 바꾸면 끝이 나지 않는 **무한소수**가 됩니다.

```
10진수: 0.1
2진수: 0.00011001100110011... (무한 반복)
```

- 컴퓨터의 그릇(FP32)은 유한하기 때문에, 이 무한한 숫자를 중간에 잘라내고 **가장 가까운 값으로 반올림**해서 저장합니다.  
- 그래서 출력창에는 `0.1`이 아니라 `0.1000000014...` 같은 지저분한 숫자가 찍히는 것입니다.

---

## 2. ✅ 왜 이번엔 True가 나왔을까? (우연의 일치)
- 보통 `0.1 + 0.2 == 0.3`은 **False**가 나오지만, 이번 실험(FP32)에서는 **True**가 나왔습니다.  
- 이유는 다음과 같습니다:
  - `0.1`의 오차 + `0.2`의 오차 결과값과  
  - `0.3` 자체가 가진 저장 오차가  
  - **우연히 같은 반올림 지점**에서 만났기 때문입니다.

```
0.1 + 0.2 결과: 0.3000000119...
0.3 자체 저장값: 0.3000000119...
```

- 두 값이 소수점 아래 20자리까지 완벽하게 일치했기 때문에 컴퓨터는 **True**라고 답한 것입니다.

---

## 3. ⚠️ 하지만 이것은 "틀린 정답"
- True가 나왔다고 해서 안심하면 안 됩니다.  
- 우리가 원한 값은 `0.3`이지만, 실제로는 **0.3000000119...**입니다.  
- 즉, **"둘 다 똑같이 틀렸기 때문에"** 비교 결과만 True가 나온 것입니다.

---

## 💡 AI와 양자화 공부에서 중요한 이유
- **오차의 누적**: AI 모델은 이런 소수점 연산을 수십억 번 반복합니다. 작은 오차도 누적되면 결과가 크게 왜곡될 수 있습니다.  
- **비교 연산의 위험성**: 실무에서는 `==` 비교를 거의 쓰지 않고,  
  ``` 
  |a - b| < ε (엡실론)
  ```  
  같은 방식으로 "거의 같다"를 판정합니다.  
- **양자화의 결단**: 소수점 대신 **정수(Integer)**로 데이터를 옮겨 이런 오차 문제를 원천 차단하거나 효율적으로 관리합니다.

---

## 🏁 최종 요약
> "0.1 + 0.2가 0.3인 이유는 둘 다 똑똑해서가 아니라,  
> 둘 다 똑같은 오차를 가진 채로 똑같은 그릇에 담겼기 때문이다.  
> 컴퓨터 세상의 소수점은 언제나 '약간의 거짓말'을 포함하고 있다!"

---


### 🎯 실습 2-1: 다양한 값에서 정밀도 손실 분석

아래 값들을 FP32, FP16, INT8로 변환하고 오차를 계산해보기

In [9]:
test_values = [0.001, 0.5, 1.234, 10.567, 100.999]

print(f"{'원본값':^12} | {'FP32':^12} | {'FP16':^12} | {'INT8':^6} | {'FP16 오차':^10}")
print("-" * 70)

for val in test_values:
    fp32_val = torch.tensor([val], dtype=torch.float32)
    fp16_val = torch.tensor([val], dtype=torch.float16)
    int8_val = torch.tensor([val], dtype=torch.int8)  # 정수형으로 바꾸면 소수점은 가차없이 버려집니다 (Floor/Truncate).

    fp16_error = abs(val - fp16_val.item())

    print(f"{val:^12.4f} | {fp32_val.item():^12.6f} | {fp16_val.item():^12.6f} | {int8_val.item():^6d} | {fp16_error:^10.2e}")

    원본값      |     FP32     |     FP16     |  INT8  |  FP16 오차  
----------------------------------------------------------------------
   0.0010    |   0.001000   |   0.001000   |   0    |  4.04e-07 
   0.5000    |   0.500000   |   0.500000   |   0    |  0.00e+00 
   1.2340    |   1.234000   |   1.234375   |   1    |  3.75e-04 
  10.5670    |  10.567000   |  10.570312   |   10   |  3.31e-03 
  100.9990   |  100.999001  |  101.000000  |  100   |  1.00e-03 



---

# 📊 데이터 타입별 손실 성적표 해석

## 1. 🧤 FP32 (완벽주의자)
- **해석**: `0.001`, `0.5`, `1.234` 모두 원본값이 거의 그대로 유지됩니다.  
- **특이점**: `100.999`에서 아주 미세하게 `...001`이 붙습니다.  
- **원인**: 컴퓨터는 2진수를 사용하기 때문에 10진수를 완벽히 표현하지 못하는 한계가 드러난 것입니다.

---

## 2. 🧀 FP16 (다이어트 중인 실수)
- **현상**:  
  - `1.234 → 1.234375`  
  - `100.999 → 101.000000` (반올림 발생)  
- **원인**: **눈금 간격이 넓기 때문**입니다. 101에 가장 가까운 눈금으로 점프해버린 것이죠.  
- **오차 특징**: 숫자가 커질수록(특히 100 단위 이상) 오차도 커집니다.

---

## 3. 🔨 INT8 (무자비한 정수 — 양자화의 원시적 형태)
- **현상**: 소수점 아래는 가차 없이 잘려 나갑니다.  
  - `0.5 → 0` (반올림 없음, 그냥 버림)  
  - `1.234 → 1`  
  - `100.999 → 100`  
- **결론**: 단순히 float를 int로 강제 변환(Casting)하면 AI 모델의 지능은 크게 손상됩니다.

---

## 💡 양자화 공부에서 얻는 교훈
- **단순 변환은 금물**: INT8로 단순 변환하면 `0.999` 같은 중요한 데이터가 사라져 AI가 바보가 됩니다.  
- **스케일링(Scaling)의 필요성**: 진짜 양자화에서는  
  - `0.5 → 5`  
  - `100.999 → 100999`  
  처럼 **곱하기(Scaling)**를 통해 소수점 데이터를 정수 범위 안으로 구출합니다.  
- **오차와의 싸움**: FP16에서 보듯, 용량을 줄이는 모든 행위는 오차를 동반합니다.  
  - 양자화의 핵심은 **"용량을 줄이면서 오차를 최소화하는 기술"**입니다.

---

## 🏁 최종 요약
> "FP16은 미세하게 흔들리고, INT8은 소수점을 도끼로 찍어내 버린다.  
> 우리가 앞으로 배울 '진짜 양자화'는 이 도끼질 대신, 데이터를 예쁘게 접어서 작은 그릇에 넣는 기술이다!"

---


---
## 실습 3: 모델 메모리 크기 계산하기

### 학습 목표
- 신경망 모델의 파라미터 수 계산
- 데이터 타입에 따른 모델 크기 예측
- 양자화로 인한 메모리 절감 효과 확인

### 3-1. 간단한 MLP 모델 생성

In [10]:
import torch.nn as nn

# [설명] 아주 간단한 인공신경망(Multi-Layer Perceptron)을 정의합니다.
# nn.Module을 상속받으면 파이토치가 알아서 "아, 이 안에 있는 변수들은 학습해야 할 파라미터구나"라고 인식합니다.
class SimpleMLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, output_size=10):
        super().__init__()
        # nn.Linear: Wx + b 연산을 수행하는 층. W(행렬)와 b(벡터)가 파라미터로 저장됩니다.
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU() # 활성화 함수 (파라미터 없음)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleMLP()
print(model)

SimpleMLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)


### 3-2. 파라미터 수 계산

In [11]:
def count_parameters(model):
    """모델의 총 파라미터 수 계산"""
    total_params = 0

    print("레이어별 파라미터 수:")
    print("=" * 50)

    # model.named_parameters(): 모델 내부의 모든 가중치(W, b)를 순회합니다.
    for name, param in model.named_parameters():
        # param.numel(): 텐서의 원소 개수 (Number of Elements) 반환
        # 예: 784x256 행렬이면 200,704개
        num_params = param.numel()
        total_params += num_params
        print(f"{name:20} | shape: {str(param.shape):20} | params: {num_params:,}")

    print("=" * 50)
    print(f"총 파라미터 수: {total_params:,}")
    return total_params

total_params = count_parameters(model)

레이어별 파라미터 수:
fc1.weight           | shape: torch.Size([256, 784]) | params: 200,704
fc1.bias             | shape: torch.Size([256])    | params: 256
fc2.weight           | shape: torch.Size([10, 256]) | params: 2,560
fc2.bias             | shape: torch.Size([10])     | params: 10
총 파라미터 수: 203,530


### 3-3. 데이터 타입별 모델 크기 계산

In [12]:
def calculate_model_size(num_params, dtype):
    """파라미터 수와 데이터 타입으로 모델 크기 계산"""
    # 각 타입별로 숫자 하나가 몇 바이트를 차지하는지 정의
    bytes_per_param = {
        'FP64': 8,
        'FP32': 4,
        'FP16': 2,
        'BF16': 2,
        'INT8': 1,
        'INT4': 0.5, # 4비트는 0.5바이트 (실제로는 2개를 1바이트에 묶어 저장)
    }

    size_bytes = num_params * bytes_per_param[dtype]
    size_kb = size_bytes / 1024
    size_mb = size_kb / 1024

    return size_bytes, size_kb, size_mb

print(f"\n모델 크기 비교 (총 파라미터: {total_params:,}개)")
print("=" * 60)

dtypes = ['FP32', 'FP16', 'BF16', 'INT8', 'INT4']
fp32_size = None

for dtype in dtypes:
    size_bytes, size_kb, size_mb = calculate_model_size(total_params, dtype)

    # 기준점(FP32) 저장
    if dtype == 'FP32':
        fp32_size = size_bytes
        ratio = 1.0
    else:
        # FP32 대비 얼마나 줄어들었는지 비율 계산
        ratio = size_bytes / fp32_size

    print(f"{dtype:6} | {size_bytes:>12,} bytes | {size_kb:>8.2f} KB | {ratio:.0%} of FP32")


모델 크기 비교 (총 파라미터: 203,530개)
FP32   |      814,120 bytes |   795.04 KB | 100% of FP32
FP16   |      407,060 bytes |   397.52 KB | 50% of FP32
BF16   |      407,060 bytes |   397.52 KB | 50% of FP32
INT8   |      203,530 bytes |   198.76 KB | 25% of FP32
INT4   |    101,765.0 bytes |    99.38 KB | 12% of FP32



---

# ⚖️ 데이터 타입별 무게 비교와 양자화의 경제학

## 1. 📏 파라미터 수
- **총 개수**: 203,530개 (약 20만 개)  
- **해석**: 모델은 가중치와 편향이라는 ‘지식 부품’으로 이루어져 있습니다.  
- **수치 분석**: 첫 번째 층(fc1)이 대부분을 차지하며, 딥러닝 모델의 크기는 보통 입력과 은닉층 사이의 **거대한 행렬 곱셈**에서 결정됩니다.  

---

## 2. 📦 데이터 타입별 무게 비교
FP32를 기준으로 압축 효과를 비교하면 다음과 같습니다:

| 데이터 타입 | 용량 | 비율 | 특징/비유 |
|-------------|------|------|-----------|
| **FP32** | 약 795KB | 100% | 숫자 하나당 4바이트. A4 용지에 글자 하나씩 큼지막하게 적는 것과 같음 |
| **FP16 / BF16** | 약 397KB | 50% | 숫자 하나당 2바이트. 종이를 반으로 접어 앞뒤로 빽빽하게 적는 셈 |
| **INT8** | 약 198KB | 25% | 숫자 하나당 1바이트. 원본의 1/4. 같은 메모리에 모델을 4개 넣을 수 있음 |
| **INT4** | 약 99KB | 12% | 숫자 하나당 0.5바이트(4비트). 원본의 1/8. 최신 LLM 실행에 가장 많이 쓰이는 방식 |

---

## 💡 양자화의 경제학
- **현실 문제**: Llama 70B 같은 거대 모델을 FP32로 돌리려면 **약 280GB VRAM**이 필요 → 일반 서버로는 불가능.  
- **해결책**: INT4로 양자화하면 **약 35GB**로 줄어듭니다 → 고성능 게이밍 그래픽카드 한두 장으로도 실행 가능.  
- **핵심**: 파라미터 개수는 그대로 두고, **데이터 타입만 정수로 줄여서** 실행 가능성을 확보하는 것이 양자화의 본질입니다.  

---

## 🏁 최종 요약
> "파라미터가 20만 개인 작은 모델도 타입을 바꾸면 용량이 800KB에서 100KB로 뚝 떨어진다.  
> 이 원리를 수십억 개의 파라미터를 가진 거대 모델에 적용하는 것이 바로 우리가 배우는 LLM 양자화의 실체다!"

---



### 🎯 실습 3-1: GPT-2 모델 크기 계산

GPT-2 모델의 파라미터 수(1.5B = 15억개)를 기준으로 각 데이터 타입별 모델 크기를 계산해보기

In [13]:
gpt2_params = 1_500_000_000  # 15억 개

print(f"GPT-2 모델 크기 비교 (파라미터: {gpt2_params:,}개)")
print("=" * 70)

dtypes = ['FP32', 'FP16', 'INT8', 'INT4']

for dtype in dtypes:
    size_bytes, size_kb, size_mb = calculate_model_size(gpt2_params, dtype)
    size_gb = size_mb / 1024 # GB 단위로 변환

    # [결과 해석]
    # FP32는 15억 * 4바이트 = 60억 바이트 = 약 5.5GB
    # INT4는 15억 * 0.5바이트 = 7.5억 바이트 = 약 0.7GB
    print(f"{dtype:6} | {size_gb:>8.2f} GB")

print("\n💡 인사이트: FP32에서 INT4로 양자화하면 모델 크기가 1/8로 줄어듭니다!")

GPT-2 모델 크기 비교 (파라미터: 1,500,000,000개)
FP32   |     5.59 GB
FP16   |     2.79 GB
INT8   |     1.40 GB
INT4   |     0.70 GB

💡 인사이트: FP32에서 INT4로 양자화하면 모델 크기가 1/8로 줄어듭니다!



---

# 🔢 GPT-2 Large (15억 파라미터)의 무게와 양자화

## 1. 파라미터 수
- **총 개수**: 1.5B (15억 개)  
- **해석**: GPT-2 Large 모델은 15억 개의 ‘지식 부품(가중치와 편향)’으로 구성되어 있습니다.  
- **핵심**: 이 숫자들을 어떤 데이터 타입(그릇)에 담느냐에 따라 모델의 **메모리 무게**가 달라집니다.  

---

## 2. 데이터 타입별 무게 비교

| 데이터 타입 | 계산 방식 | 용량 | 특징/해석 |
|-------------|-----------|------|-----------|
| **FP32** | 1.5B × 4바이트 | 약 5.59 GB | 성능은 최고지만 너무 무겁습니다. 최소 6GB VRAM 필요. |
| **FP16** | 1.5B × 2바이트 | 약 2.79 GB | 정밀도를 조금 포기해 용량이 반으로 줄어듭니다. 4GB GPU에서도 실행 가능. |
| **INT4** | 1.5B × 0.5바이트(4비트) | 약 0.70 GB | 원본 대비 1/8 무게. 스마트폰이나 저사양 PC에서도 실행 가능. |

---

## 💡 양자화 공부의 핵심 포인트
- **VRAM 한계 극복**: 그래픽카드 메모리는 추가 장착이 불가능한 자원. 양자화는 이 제한된 공간에 더 큰 모델을 담을 수 있게 해줍니다.  
- **속도의 기적**: 모델이 가벼워지면 메모리 전송 속도도 빨라져, 답변 속도 자체가 개선됩니다.  
- **실무적 가치**: Llama 같은 대형 모델도 4비트 양자화 덕분에 일반 PC에서 실행 가능.  

---

## 🏁 최종 요약
> "15억 개의 파라미터를 4비트(INT4)로 깎는 순간,  
> 고성능 서버에서만 돌아가던 AI가 내 주머니 속 스마트폰에서도 돌아가는  
> **'모두의 AI'**로 변신하게 된다!"

---


### 3-4. 실제 PyTorch 모델 메모리 측정

In [14]:
def get_model_size(model):
    """실제 모델 객체가 메모리에서 차지하는 크기 측정"""
    param_size = 0
    buffer_size = 0

    # 1. 학습 가능한 파라미터(Weights) 크기 누적
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    # 2. 버퍼(Buffer) 크기 누적 (예: BatchNorm의 running_mean 등, 학습되진 않지만 저장되는 값)
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    total_size = param_size + buffer_size
    return total_size

# FP32 모델 (기본값)
model_fp32 = SimpleMLP()
size_fp32 = get_model_size(model_fp32)

# FP16 모델
# .half(): 모델 내부의 모든 파라미터를 FP32 -> FP16으로 캐스팅합니다.
model_fp16 = SimpleMLP().half() 
size_fp16 = get_model_size(model_fp16)

# BF16 모델
# .to(dtype): 모델을 특정 타입으로 변환합니다.
model_bf16 = SimpleMLP().to(torch.bfloat16)
size_bf16 = get_model_size(model_bf16)

print("실제 PyTorch 모델 메모리 크기")
print("=" * 50)
print(f"FP32 모델: {size_fp32:,} bytes ({size_fp32/1024:.2f} KB)")
print(f"FP16 모델: {size_fp16:,} bytes ({size_fp16/1024:.2f} KB)")
print(f"BF16 모델: {size_bf16:,} bytes ({size_bf16/1024:.2f} KB)")
print(f"\n메모리 절감률: {(1 - size_fp16/size_fp32) * 100:.1f}%")


실제 PyTorch 모델 메모리 크기
FP32 모델: 814,120 bytes (795.04 KB)
FP16 모델: 407,060 bytes (397.52 KB)
BF16 모델: 407,060 bytes (397.52 KB)

메모리 절감률: 50.0%



---

# 🧪 get_model_size 함수의 정체 (진짜 무게 재기)

## 1. 함수의 역할
- **parameters() (학습하는 지식)**: AI가 학습을 통해 채워 넣는 핵심 가중치(Weights).  
- **buffers() (보조 지식)**: 학습하지는 않지만 모델이 작동하는 데 필요한 보조 데이터.  
- **nelement() × element_size()**:  
  - `nelement()`: 재료(데이터)가 몇 개 들어갔는지  
  - `element_size()`: 재료 하나가 몇 바이트인지  
  - 두 값을 곱해 실제 메모리 무게를 계산합니다.  
- **핵심**: 이 함수는 운영체제에게 직접 물어봐서 "실제로 몇 칸을 차지하고 있나?"를 알려주는 **진짜 무게 측정기**입니다.

---

## 2. 📦 결과 분석: "딱 절반의 마법"
- **FP32 (약 795 KB)**: 숫자 하나당 4바이트 사용.  
- **FP16 / BF16 (약 397 KB)**: 숫자 하나당 2바이트 사용.  
- **해석**: 4바이트 → 2바이트로 바꾸니 메모리 사용량이 정확히 **1/2**로 줄었습니다.  
- **의미**: 이론과 실제가 완벽하게 일치하는 순간을 보여줍니다.

---

## 💡 실전에서 중요한 이유
- **필수 다이어트 기술**: 모델 배포 시 가장 먼저 쓰이는 메모리 절감 방법.  
- **.half() (FP16 변환)**:  
  - 코드 한 줄(`.half()`)만으로 모델 무게를 즉시 50% 줄임.  
  - 예: "그래픽카드가 8GB인데 모델이 12GB라 안 돌아가요!" → `.half()` 적용 시 6GB로 줄어 실행 가능.  
- **BF16의 경제성**:  
  - FP16과 용량은 동일하지만, 숫자가 안정적으로 유지됨.  
  - 즉, **"똑같이 가벼우면서 더 튼튼한"** 모델을 만들 수 있음.

---

## 🏁 최종 요약
> "이론상 50%가 줄어들 것이라는 예상은 실제 PyTorch 메모리상에서도  
> 1바이트의 오차 없이 정확히 50.0% 절감으로 나타났다.  
> `.half()`나 `.to(dtype)` 같은 명령어 하나가 수백억 원짜리 장비를 아껴주는 마법이 된다!"

---



---
### 메모리 절감 효과
| 변환 | 메모리 절감 |
|------|------------|
| FP32 → FP16 | 50% 절감 |
| FP32 → INT8 | 75% 절감 |
| FP32 → INT4 | 87.5% 절감 |


### 📊 Part 1 양자화 기초 실습: 결과 해석 및 분석

이 문서는 `quantization_basics_lab.py` 코드를 실행했을 때 나오는 결과들을 심층 분석합니다.  
단순히 **"숫자가 다르네?"**를 넘어 **"왜 컴퓨터는 이렇게밖에 못할까?"** 를 이해하는 것이 핵심입니다.

---

### 1. 🔍 FP16의 충격적인 정밀도 부족 (실습 1-1 해석)

실습 코드에서 `x = 2048.0`일 때 1을 더해봤습니다. 결과가 어땠나요?

- **예상:** 2049, 2050, 2051... 이렇게 늘어나야 함  
- **실제 (FP16):** 2048.0 에서 멈춤! (또는 2050으로 점프)  

💡 **왜 이런 일이 발생하나요? (Why)**  
FP16(16비트 실수)은 숫자를 저장할 공간이 아주 좁습니다.  

- **지수(Exponent, 5bit):** 숫자의 스케일(크기) 담당  
- **가수(Mantissa, 10bit):** 숫자의 디테일 담당  

숫자가 커지면(지수가 커지면), 숫자 사이의 간격(Step)도 같이 커집니다.  
- 0 근처에서는 0.0001 단위로 표현 가능하지만,  
- 2048 근처에서는 표현 가능한 최소 간격이 1보다 커질 수 있습니다.  

마치 자(Ruler)의 눈금이 처음엔 1mm였다가, 뒤로 갈수록 1cm, 1m로 듬성듬성해지는 것과 같습니다.  

**결론:** FP16으로 학습할 때 값이 너무 커지면(Gradient Explosion 등), 미세한 업데이트(+0.001)는 전부 씹혀서 학습이 안 될 수 있습니다.

---

### 2. 🧠 BF16 vs FP16 (실습 2-2 해석)

파이(π) 값을 저장했을 때의 오차 비교입니다.

| 타입   | 오차 수준       | 특징 |
|--------|----------------|------|
| FP32   | \(10^{-7}\) (매우 작음) | 표준. 딥러닝의 기준점 |
| FP16   | \(10^{-4}\) (작음)     | 가수부가 10비트라 그나마 디테일함 |
| BF16   | \(10^{-2}\) (큼)       | 가수부가 7비트라 디테일이 많이 깨짐. 3.14... 뒤는 거의 틀림 |

💡 **BF16이 오차가 더 큰데 왜 구글/엔비디아는 BF16을 쓸까요?**  
- BF16은 "디테일(가수)"을 포기하고 "범위(지수)"를 챙긴 타입입니다.  
- **FP16의 문제:** 65,504를 넘어가면 숫자가 터짐 (Overflow → Infinity). 학습 중단됨.  
- **BF16의 해결:** FP32와 똑같은 범위(\(10^{38}\))까지 표현 가능.  
  → "좀 틀려도 되니까 터지지만 마라!"  

딥러닝 학습에서는 소수점 4번째 자리의 정확성보다, 학습이 안 터지고 계속 돌아가는 것이 훨씬 중요하기 때문입니다.

---

### 3. 📉 메모리 다이어트 효과 (실습 3-3 해석)

GPT-2 (15억 파라미터) 모델을 메모리에 올린다고 가정했을 때의 용량 차이입니다.

| 데이터 타입 | 모델 크기 (GB) | 비유 |
|-------------|----------------|------|
| FP32 (원본) | 5.59 GB        | 큰 캐리어 가방 💼 |
| FP16 (Half) | 2.79 GB        | 배낭 🎒 |
| INT8 (Quant)| 1.40 GB        | 핸드백 👜 |
| INT4 (Quant)| 0.70 GB        | 주머니 👛 |

💡 **인사이트**  
FP32 → INT4로 가면 용량이 **1/8로 줄어듭니다.**  
80GB짜리 초거대 모델(Llama-3 70B)도 INT4로 줄이면  
- 40GB(FP16) → 20GB(INT4) 가 되어,  
- 일반인도 **3090/4090 GPU 1장으로 돌려볼 수 있게** 되는 것입니다.  

이것이 바로 우리가 **양자화(Quantization)** 를 배우는 이유입니다.

---

### 4. ✅ 최종 요약 (Takeaway)

- **비트(Bit)는 돈이다:** 비트 수를 줄이면 메모리 비용이 기하급수적으로 줍니다.  
- **세상에 공짜는 없다:** 비트를 줄이면 정밀도(Precision)가 떨어지거나, 표현 범위(Range)가 줄어듭니다.  
- **적절한 타협:** 딥러닝 모델은 생각보다 둔감해서, **INT4(4비트)** 정도의 거친 해상도로도 충분히 똑똑하게 대답합니다.  

*(Part 2에서 계속!)*


---

# 📋 실무 고수들의 데이터 타입 선택 공식

## 상황별 추천 타입

| 상황 | 추천 타입 | 이유 (실무 포인트) |
|------|-----------|---------------------|
| **모델 학습 (Training)** | **BF16** | 학습 중 에러 방지. 숫자가 커져서 터지는 일(Overflow)이 거의 없음. FP32와 동일한 범위를 가져 안정적임. |
| **모델 배포 (Inference)** | **FP16** | 이미 학습이 끝났으니 안정성보다는 정밀도가 중요. 말 한마디를 더 예쁘게 표현 가능. 호환성도 뛰어남. |
| **오래된 GPU (V100 이하)** | **FP16** | 선택의 여지 없음. 구형 GPU는 BF16을 지원하지 않음. |
| **최신 고성능 GPU (A100/H100)** | **BF16** | 속도도 빠르고 학습 성능이 훨씬 안정적. 최신 GPU에서는 BF16이 최적. |

---

## 💡 실무 상황별 한 문장 가이드
- **학습 중 에러 발생 (0이나 Inf가 뜸)** → FP16 대신 **BF16**으로 전환하세요.  
- **서버비 절감, 모델 용량 반으로 줄이고 싶음** → `.half()` 명령어로 모든 파라미터를 **FP16**으로 변환하세요.  
- **용량은 줄이되, 정밀도는 최대한 유지하고 싶음** → 배포 시에는 **FP16**이 유리합니다. (가수부 10비트로 더 세밀한 표현 가능)  

---

## 🏁 최종 요약: "딱 이것만 기억하세요"
- **학습(만들 때)** → 안전한 **BF16** (안 터지는 게 최고)  
- **추론(사용할 때)** → 정밀한 **FP16** (조금이라도 더 똑똑하게)  
- 단, **RTX 30 시리즈 이상 또는 A100/H100** GPU에서만 BF16 사용 가능. 구형 GPU는 FP16만 지원.  

---
